# Traitement automatisé — toutes les visites (V0, V1, V3, V5, Vc)



In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce
import re

In [38]:
def construire_tableau_synthetique(result, v):
    suffix = f"_V{v}"
    dfs = []

    # 1. Feuille Vx
    df_v = result[f"V{v}"].copy()
    df_v = df_v.drop(columns=[c for c in df_v.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
    df_v = df_v.rename(columns={c: f"{c}{suffix}" for c in df_v.columns if c != "SUBJID"})
    dfs.append(df_v)

    # 2. PDQ39_SI uniquement
    if "PDQ39" in result:
        df = result["PDQ39"][["SUBJID", "PDQ39_SI"]].copy()
        df = df.rename(columns={"PDQ39_SI": f"PDQ39_SI{suffix}"})
        dfs.append(df)

    # 3. UPDRSIII_S pour V0, sinon UPDRSIII
    key = "UPDRSIII" if (v == 0 and "UPDRSIII" in result) else "UPDRSIII"
    if key in result:
        df = result[key].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        # if key =="UPDRSIII" and v != 1 :
        #     df=df.drop(columns=["UPDRSIII_tot","statut"])
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 4. UPDRSIV
    if "UPDRSIV" in result:
        df = result["UPDRSIV"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 5. ECMP
    if "ECMP" in result:
        df = result["ECMP"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 6. QUIP
    if "QUIP" in result:
        df = result["QUIP"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 7. LARS
    if "LARS" in result:
        df = result["LARS"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT","LARS_SCORE","statut","LARS_RESULTAT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 8. UPPS
    if "UPPS" in result:
        df = result["UPPS"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 9. DIGITSMT_TRAILMT_DKEFS
    if "DIGITSMT_TRAILMT_DKEFS" in result:
        df = result["DIGITSMT_TRAILMT_DKEFS"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 10. HAMD_tot uniquement
    if "HAMD" in result:
        df = result["HAMD"][["SUBJID", "HAMD_tot"]].copy()
        df = df.rename(columns={"HAMD_tot": f"HAMD_tot{suffix}"})
        dfs.append(df)

    # 11. HAMA_tot uniquement
    if "HAMA" in result:
        df = result["HAMA"][["SUBJID", "HAMA_tot"]].copy()
        df = df.rename(columns={"HAMA_tot": f"HAMA_tot{suffix}"})
        dfs.append(df)

    # 12. MOCA_tot uniquement
    if "MOCA" in result:
        df = result["MOCA"][["SUBJID", "MOCA_tot"]].copy()
        df = df.rename(columns={"MOCA_tot": f"MOCA_tot{suffix}"})
        dfs.append(df)

    # 13. PSYCHOTROPES → colonnes CLASS* uniquement
    if "PSYCHOTROPES" in result:
        df = result["PSYCHOTROPES"][["SUBJID",'Anxiolytiques', 'Antidépresseurs','Neuroleptiques', 'Thymorégulateurs']].copy()
        df = df.rename(columns={
        'Anxiolytiques': f"Anxiolytiques{suffix}",
        'Antidépresseurs': f"Antidépresseurs{suffix}",
        'Neuroleptiques': f"Neuroleptiques{suffix}",
        'Thymorégulateurs': f"Thymorégulateurs{suffix}"
        })
        dfs.append(df)
        

    # 14. AUTRE_PARKINSON → colonnes CLASS* uniquement
    # if "AUTRE_PARKINSON" in result:
    #     df = result["AUTRE_PARKINSON"].copy()
    #     cols = ["SUBJID"] + [c for c in df.columns if c.upper().startswith("CLASS")]
    #     df = df[cols]
    #     df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
    #     dfs.append(df)

    # 15. LEDD
    if "LEDD" in result:
        df = result["LEDD"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)
    
    if "FREQUENCE" in result :
        df = result["FREQUENCE"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # ── Fusion sur SUBJID ─────────────────────────────────────────────
    df_final = dfs[0]
    for df_next in dfs[1:]:
        df_final = pd.merge(df_final, df_next, on="SUBJID", how="outer")

    return df_final

---
## Fonction principale : `traiter_visite(v, vc=False)`

In [39]:
def traiter_visite(v, vc=False):

    chemin = f"Output/version_3/V{v}.xlsx"
    print(f"\n{'='*60}")
    print(f"  TRAITEMENT  V{v}  —  {chemin}")
    print(f"{'='*60}")

    # ==================================================================
    # FEUILLES COMMUNES  (présentes dans toutes les visites, y compris Vc)
    # ==================================================================
     
    df_V = pd.read_excel(chemin, sheet_name=f"V{v}")
    # df_V = nettoyer_codes_manquants(df_V)

    # ── LEDD ──────────────────────────────────────────────────────────
    df_LEDD = pd.read_excel(chemin, sheet_name="LEDD")
    # df_LEDD["somme_calc"] = df_LEDD.iloc[:, 2:-1].sum(axis=1, skipna=True)
    # df_LEDD["statut"] = np.where(
    #     np.isclose(df_LEDD["somme_calc"], df_LEDD["ledd_tot"], atol=0.01),
    #     "ok", "différent"
    # )
    # df_LEDD["ledd_tot"] = df_LEDD["somme_calc"]
    # df_LEDD.drop(columns=["somme_calc", "statut"], inplace=True)
    # df_Total_LEDD = df_LEDD[["SUBJID", "ledd_tot"]]
    print(f"df_LEDD                 : {df_LEDD.shape}")

    # ── PSYCHOTROPES ──────────────────────────────────────────────────
    df_PSYCHOTROPES = pd.read_excel(chemin, sheet_name="PSYCHOTROPES")
    # print(f"\nFeuille PSYCHOTROPES    : {df_PSYCHOTROPES.shape}")
    # df_PSYCHOTROPES = nettoyer_codes_manquants(df_PSYCHOTROPES)
    # cols = df_PSYCHOTROPES.columns[:2].tolist()
    # cols += [c for c in df_PSYCHOTROPES.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    # df_PSYCHOTROPES = df_PSYCHOTROPES[cols]
    # df_PSYCHOTROPES = classer_medicaments(df_PSYCHOTROPES, dict_complet)
    print(f"df_PSYCHOTROPES         : {df_PSYCHOTROPES.shape}")

    # ── AUTRE_PARKINSON ───────────────────────────────────────────────
    df_AUTRE_PARKINSON = pd.read_excel(chemin, sheet_name="AUTRE_PARKINSON")
    # df_AUTRE_PARKINSON = nettoyer_codes_manquants(df_AUTRE_PARKINSON)
    # cols = df_AUTRE_PARKINSON.columns[:2].tolist()
    # cols += [c for c in df_AUTRE_PARKINSON.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    # df_AUTRE_PARKINSON = df_AUTRE_PARKINSON[cols]
    # df_AUTRE_PARKINSON = classer_medicaments(df_AUTRE_PARKINSON, dict_complet)
    print(f"df_AUTRE_PARKINSON      : {df_AUTRE_PARKINSON.shape}")

    # ── CONSO_SPECIFIQUE ──────────────────────────────────────────────
    df_CONSO_SPECIFIQUE = pd.read_excel(chemin, sheet_name="CONSO_SPECIFIQUE")
    # df_CONSO_SPECIFIQUE = nettoyer_codes_manquants(df_CONSO_SPECIFIQUE)
    # cols = df_CONSO_SPECIFIQUE.columns[:2].tolist()
    # cols += [c for c in df_CONSO_SPECIFIQUE.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    # df_CONSO_SPECIFIQUE = df_CONSO_SPECIFIQUE[cols]
    print(f"df_CONSO_SPECIFIQUE     : {df_CONSO_SPECIFIQUE.shape}")

    # ==================================================================
    # CAS SPÉCIAL  →  Vc  (uniquement les feuilles communes)
    # ==================================================================
    # if vc:
    #     return {
    #         f"V{v}"             : df_V,
    #         "LEDD"             : df_LEDD,
    #         "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
    #         "PSYCHOTROPES"     : df_PSYCHOTROPES,
    #         "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
    #     }
        
    if vc:
        result = {
            f"V{v}"             : df_V,
            "LEDD"             : df_LEDD,
            "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
            "PSYCHOTROPES"     : df_PSYCHOTROPES,
            "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
        }

        result["SYNTHESE"] = construire_tableau_synthetique(result, v)

        return result
    # ==================================================================
    # FEUILLES SPÉCIFIQUES  V0 → V5
    # ==================================================================

    # ── UPDRSIII ──────────────────────────────────────────────────────
    if v == 0:
        df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII_S")
        print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")

    elif v in (1, 3, 5):
        df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
        print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")
        

    # ── UPDRSIV ───────────────────────────────────────────────────────
    df_UPDRSIV = pd.read_excel(chemin, sheet_name="UPDRSIV")
    df_UPDRSIV.drop("MDSUPDRSPARTIE4",axis=1,inplace=True)
    df_UPDRSIV.drop("UPDRSIV_tot",axis=1)
    
    # ── PDQ39 ─────────────────────────────────────────────────────────
    df_PDQ39 = pd.read_excel(chemin, sheet_name="PDQ39")
    print(f"\nFeuille PDQ39           : {df_PDQ39.shape}")

    # ── QUIP ──────────────────────────────────────────────────────────
    df_QUIP = pd.read_excel(chemin, sheet_name="QUIP")
    print(f"\nFeuille QUIP            : {df_QUIP.shape}")


    # ── MOCA ──────────────────────────────────────────────────────────
    df_MOCA = pd.read_excel(chemin, sheet_name="MOCA")
    df_MOCA = df_MOCA[["SUBJID","MOCA_tot"]]
    print(f"\nFeuille MOCA            : {df_MOCA.shape}")


    # ── HAMA ──────────────────────────────────────────────────────────
    df_HAMA = pd.read_excel(chemin, sheet_name="HAMA")
    df_HAMA = df_HAMA[["SUBJID","HAMA_tot"]]
    print(f"df_HAMA                 : {df_HAMA.shape}")

    # ── HAMD ──────────────────────────────────────────────────────────
    df_HAMD = pd.read_excel(chemin, sheet_name="HAMD")
    df_HAMD = df_HAMD[["SUBJID", "HAMD_tot"]]
    print(f"df_HAMD                 : {df_HAMD.shape}")

    # ── LARS ──────────────────────────────────────────────────────────
    df_LARS = pd.read_excel(chemin, sheet_name="LARS")
    df_LARS.drop(columns=["LARS_tot","statut"],inplace=True)
    # df_LARS = df_LARS.iloc[:, :-3]
    print(f"df_LARS                 : {df_LARS.shape}")

    # ── ECMP ──────────────────────────────────────────────────────────


    df_ECMP = pd.read_excel(chemin, sheet_name="ECMP")
    cols_atcd = [col for col in df_ECMP.columns if col.endswith('ATCD')]
    df_ECMP = df_ECMP.drop(columns=cols_atcd)
    print(f"\nFeuille ECMP            : {df_ECMP.shape}")
    

    # ── DIGITSMT_TRAILMT_DKEFS ────────────────────────────────────────
    df_DIGITSMT = pd.read_excel(chemin, sheet_name="DIGITSMT_TRAILMT_DKEFS")
    print(f"\nFeuille DIGITSMT        : {df_DIGITSMT.shape}")


    # ── UPPS (fichier externe, V0 V1 V3 V5) ──────────────────────────
    if v in (0, 1, 3, 5):
        df_UPPS = pd.read_excel(chemin, sheet_name="UPPS")
        print(f"df_UPPS                 : {df_UPPS.shape}")

    # ── FREQUENCE  (V1, V2, V3) ───────────────────────────────────────
    if v in (1, 2, 3):
        df_FREQUENCE = pd.read_excel(chemin, sheet_name="FREQUENCE")



    # ==================================================================
    # CONSTRUCTION DU DICTIONNAIRE DE RETOUR
    # ==================================================================
    result = {
        f"V{v}"                   : df_V,
        "LEDD"                    : df_LEDD,
        "CONSO_SPECIFIQUE"        : df_CONSO_SPECIFIQUE,
        "PSYCHOTROPES"            : df_PSYCHOTROPES,
        "AUTRE_PARKINSON"         : df_AUTRE_PARKINSON,
        "UPDRSIII"                : df_UPDRSIII,
        "UPDRSIV"                 : df_UPDRSIV,
        "PDQ39"                   : df_PDQ39,
        "QUIP"                    : df_QUIP,
        "MOCA"                    : df_MOCA,
        "HAMA"                    : df_HAMA,
        "HAMD"                    : df_HAMD,
        "LARS"                    : df_LARS,
        "ECMP"                    : df_ECMP,
        "DIGITSMT_TRAILMT_DKEFS"  : df_DIGITSMT,
    }



    if v in (0, 1, 3, 5):
        result["UPPS"] = df_UPPS

    if v in (1, 2, 3):
        result["FREQUENCE"] = df_FREQUENCE
    
    # ── SYNTHESE ──────────────────────────────────────────────────────
    result["SYNTHESE"] = construire_tableau_synthetique(result, v)


    return result

In [40]:
ORDRE_FEUILLES = [
    
    "PDQ39",
    "UPDRSIII",
    "UPDRSIV",  
    "ECMP",
    "QUIP",
    "LARS",
    "UPPS",
    "HAMD",
    "HAMA",
    "MOCA",
    "DIGITSMT_TRAILMT_DKEFS",

    "FREQUENCE",
    
    "CONSO_SPECIFIQUE",
    "PSYCHOTROPES",
    "AUTRE_PARKINSON",
    
    "LEDD",
    "FREQUENCE"
    "SYNTHESE"
]

def reordonner_feuilles(sheets, v):
    cle_visite    = f"V{v}"
    ordre_complet = [cle_visite] + ORDRE_FEUILLES
    return {k: sheets[k] for k in ordre_complet if k in sheets}

---
## Utilitaire d'écriture Excel

In [41]:
def ecrire_excel(sheets_dict, version, output_dir="Output/version_4"):
    filepath = f"{output_dir}/V{version}.xlsx"
    with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
        for sheet_name, df in sheets_dict.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"[OK] {filepath}  →  {len(sheets_dict)} feuilles")

---
## Traitement + écriture de chaque visite

### Vc

In [42]:
sheets_Vc = traiter_visite("c", vc=True)
sheets_Vc = reordonner_feuilles(sheets_Vc, "c")
ecrire_excel(sheets_Vc, "c")


  TRAITEMENT  Vc  —  Output/version_3/Vc.xlsx
df_LEDD                 : (491, 7)
df_PSYCHOTROPES         : (835, 46)
df_AUTRE_PARKINSON      : (835, 46)
df_CONSO_SPECIFIQUE     : (835, 28)
[OK] Output/version_4/Vc.xlsx  →  5 feuilles


### V0

In [43]:
sheets_V0 = traiter_visite(0)
sheets_V0 = reordonner_feuilles(sheets_V0, 0)

ecrire_excel(sheets_V0, 0)



  TRAITEMENT  V0  —  Output/version_3/V0.xlsx
df_LEDD                 : (794, 7)
df_PSYCHOTROPES         : (835, 46)
df_AUTRE_PARKINSON      : (835, 46)
df_CONSO_SPECIFIQUE     : (835, 28)

Feuille UPDRSIII        : (836, 89)

Feuille PDQ39           : (835, 53)

Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 2)
df_HAMA                 : (835, 2)
df_HAMD                 : (835, 2)
df_LARS                 : (835, 16)

Feuille ECMP            : (835, 23)

Feuille DIGITSMT        : (835, 21)
df_UPPS                 : (835, 22)
[OK] Output/version_4/V0.xlsx  →  16 feuilles


### V1

In [44]:
sheets_V1 = traiter_visite(1)
sheets_V1 = reordonner_feuilles(sheets_V1, 1)
ecrire_excel(sheets_V1, 1)


  TRAITEMENT  V1  —  Output/version_3/V1.xlsx
df_LEDD                 : (525, 7)
df_PSYCHOTROPES         : (835, 46)
df_AUTRE_PARKINSON      : (835, 46)
df_CONSO_SPECIFIQUE     : (835, 28)

Feuille UPDRSIII        : (836, 155)

Feuille PDQ39           : (835, 53)

Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 2)
df_HAMA                 : (835, 2)
df_HAMD                 : (835, 2)
df_LARS                 : (835, 16)

Feuille ECMP            : (835, 23)

Feuille DIGITSMT        : (835, 21)
df_UPPS                 : (835, 22)
[OK] Output/version_4/V1.xlsx  →  17 feuilles


### V3

In [45]:
sheets_V3 = traiter_visite(3)
sheets_V3 = reordonner_feuilles(sheets_V3, 3)
ecrire_excel(sheets_V3, 3)


  TRAITEMENT  V3  —  Output/version_3/V3.xlsx
df_LEDD                 : (272, 7)
df_PSYCHOTROPES         : (835, 46)
df_AUTRE_PARKINSON      : (835, 46)
df_CONSO_SPECIFIQUE     : (835, 28)

Feuille UPDRSIII        : (835, 43)

Feuille PDQ39           : (835, 53)

Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 2)
df_HAMA                 : (835, 2)
df_HAMD                 : (835, 2)
df_LARS                 : (835, 16)

Feuille ECMP            : (835, 23)

Feuille DIGITSMT        : (835, 21)
df_UPPS                 : (835, 22)
[OK] Output/version_4/V3.xlsx  →  17 feuilles


### V5

In [46]:
sheets_V5 = traiter_visite(5)
sheets_V5 = reordonner_feuilles(sheets_V5, 5)
ecrire_excel(sheets_V5, 5)


  TRAITEMENT  V5  —  Output/version_3/V5.xlsx
df_LEDD                 : (313, 7)
df_PSYCHOTROPES         : (835, 46)
df_AUTRE_PARKINSON      : (835, 46)
df_CONSO_SPECIFIQUE     : (835, 28)

Feuille UPDRSIII        : (835, 43)

Feuille PDQ39           : (835, 53)

Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 2)
df_HAMA                 : (835, 2)
df_HAMD                 : (835, 2)
df_LARS                 : (835, 16)

Feuille ECMP            : (835, 23)

Feuille DIGITSMT        : (835, 21)
df_UPPS                 : (835, 22)
[OK] Output/version_4/V5.xlsx  →  16 feuilles


#

# Prétraitement info statiques 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
info =  pd.read_excel("Output/version_2/info.xlsx")
info.head()


,SUBJID,INIT_PAT,D_SCREEN,D_1ER_SYMPT,D_DIAG,D_LDOPA,D_TTT_DOPAM,D_FLUCTU_MOTR,D_FLUCTU_NONMOTR,D_DYSKINESIE,...,AGE,SEXE,SUIVI_ETUDE,D_FIN_ETUDE,D_SORTIE_PREMA,MOTIF_SORTIE_PREMA,AUTRE_PRECIS,D_INVESTIGATEUR,NOM_INVESTIGATEUR,MOTIF_SORTIE_PREMA1
0,Subject Identifier for the Study,initiales patient,date de la visite de screening,année premiers symptomes,année du diagnostic de la maladie,année d'introduction de la L-DOPA,année d'introduction du traitement dopaminergique,année d'apparition des fluctuations motrices,année d'apparition des fluctuations non motrices,année d'apparition des dyskinésies,...,âge,sexe,suivi étude,date de fin de l'étude,date de sortie prématurée,motif de sortie prématurée,autre précision,date de signature de l'investigateur,nom de l'investigateur,motif de sortie prématurée LIB
1,01-001,SR,18/11/2013,1998,1999,2000,2000,2002,2000,.D,...,67,1,0,NaN,22/11/2013,1,NaN,18/03/2014,MOREAU CAROLINE,Patient non opéré
2,01-002,TM,13/01/2014,2006,2006,2006,2006,2011,.K,.K,...,59,1,0,NaN,20/02/2014,6,SUSPICION DE CANCER PULMONAIRE,24/03/2014,DR MOREAU,Autre
3,01-003,SJ,04/03/2014,2001,2001,2003,2001,2004,.K,2004,...,61,1,0,NaN,07/03/2014,1,NaN,21/03/2014,DR HOPES LUCIE,Patient non opéré
4,01-004,DJ,12/05/2014,1998,2000,2003,2000,2003,2003,2003,...,65,2,0,NaN,09/01/2015,6,RETRAIT DU MATERIEL ET RETRAIT DE CONSENTEMENT,09/01/2015,DEVOS,Autre


In [12]:
info.drop("D_SCREEN",axis=1,inplace=True)
info = nettoyer_codes_manquants(info)
info.to_excel("Output/version_3/info.xlsx",index=False)


  → codes remplacés par NaN : ['.K', '.D', '.A', '.C']


# Fusionner 

In [24]:
def reordonner_colonnes_par_variable(df):
    """
    Réorganise les colonnes du DataFrame fusionné de façon à grouper
    chaque variable ensemble sur toutes les visites.
    
    Ex: poids_V0, poids_V1, poids_V3 | taille_V0, taille_V1 | PDQ39_V0, PDQ39_V1 ...
    """
    import re

    visites = ["V0", "VC", "V1", "V3", "V5"]  # ordre des visites voulu

    # Séparer SUBJID du reste
    autres_cols = [c for c in df.columns if c != "SUBJID"]

    # Extraire le suffixe de visite d'une colonne
    def extraire_base_et_visite(col):
        for v in sorted(visites, key=len, reverse=True):  # tester les plus longs d'abord
            if col.endswith(f"_{v}"):
                base = col[:-(len(v) + 1)]  # retire "_Vx"
                return base, v
        return col, None  # colonne sans suffixe visite

    # Construire un dict : base → {visite: colonne}
    from collections import defaultdict, OrderedDict
    groupes = defaultdict(dict)
    sans_visite = []

    for col in autres_cols:
        base, visite = extraire_base_et_visite(col)
        if visite:
            groupes[base][visite] = col
        else:
            sans_visite.append(col)

    # Reconstruire l'ordre des colonnes :
    # Pour chaque base (dans l'ordre d'apparition original), 
    # mettre V0, VC, V1, V3, V5 côte à côte
    ordre_bases = list(dict.fromkeys(
        extraire_base_et_visite(c)[0] for c in autres_cols
        if extraire_base_et_visite(c)[1] is not None
    ))

    colonnes_ordonnees = ["SUBJID"]
    for base in ordre_bases:
        for v in visites:
            if v in groupes[base]:
                colonnes_ordonnees.append(groupes[base][v])

    # Ajouter les colonnes sans suffixe visite à la fin
    colonnes_ordonnees += sans_visite

    # Garder seulement les colonnes qui existent vraiment
    colonnes_ordonnees = [c for c in colonnes_ordonnees if c in df.columns]

    return df[colonnes_ordonnees]

In [25]:
def construire_df_final(dossier="."):
    """
    Lit directement les fichiers Excel de chaque visite,
    construit le tableau synthétique pour chaque visite,
    merge tout et réorganise les colonnes.
    """
    visites = [
        ("v0.xlsx", 0),
        ("vc.xlsx", "c"),
        ("v1.xlsx", 1),
        ("v3.xlsx", 3),
        ("v5.xlsx", 5),
    ]

    dfs = []
    for fichier, v in visites:
        chemin = os.path.join(dossier, fichier)
        if not os.path.exists(chemin):
            print(f"Fichier manquant : {fichier}, ignoré.")
            continue
        
        # Lire toutes les feuilles d'un coup → dict {nom_feuille: DataFrame}
        result = pd.read_excel(chemin, sheet_name=None)
        
        df_v = construire_tableau_synthetique(result, v)
        dfs.append(df_v)

    # Merge tous les Vx sur SUBJID
    from functools import reduce
    df_all = reduce(lambda l, r: pd.merge(l, r, on="SUBJID", how="outer"), dfs)

    # Réorganiser les colonnes par variable
    df_final = reordonner_colonnes_par_variable(df_all)

    return df_final

In [26]:
import os
import pandas as pd

df_final = construire_df_final(dossier="Output/version_4")
df_final.to_excel("data02.xlsx", index=False)

In [32]:
df_final.head()

,SUBJID,DATE_V0,DATE_V1,DATE_V3,DATE_V5,POIDS_V0,POIDS_V1,POIDS_V3,POIDS_V5,TAILLE_V0,...,DATE_Vc,Anxiolytiques_Vc,Antidépresseurs_Vc,Neuroleptiques_Vc,Thymorégulateurs_Vc,ledd_levot_Vc,ledd_agot_Vc,ledd_imaobt_Vc,ledd_mantt_Vc,ledd_tot_Vc
0,01-001,18/11/2013,NaN,NaN,NaN,76.0,NaN,NaN,NaN,171.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
1,01-002,13/01/2014,NaN,NaN,NaN,88.0,NaN,NaN,NaN,180.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,01-003,04/03/2014,NaN,NaN,NaN,89.0,NaN,NaN,NaN,176.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3,01-004,12/05/2014,NaN,NaN,NaN,59.0,NaN,NaN,NaN,163.0,...,23/09/2014,2.0,1.0,0.0,0.0,700.00,120.0,NaN,NaN,820.00
4,01-005,16/06/2014,25/01/2016,17/04/2018,NaN,81.0,82.0,NaN,NaN,169.0,...,27/01/2015,1.0,0.0,0.0,0.0,893.75,NaN,NaN,NaN,893.75


In [33]:
df_info = pd.read_excel("Output/version_3/info.xlsx")

df_complet = pd.merge(df_info, df_final, on="SUBJID", how="outer")

# Mettre SUBJID en premier
cols = ["SUBJID"] + [c for c in df_complet.columns if c != "SUBJID"]
df_complet = df_complet[cols]

df_complet.to_excel("Output/version_3/data01.xlsx", index=False)